In [1]:
import pandas as pd
import sys
from pathlib import Path

# 현재 경로부터 상위 폴더를 탐색
current = Path.cwd().resolve()

ROOT_DIR = next(
    (
        path for path in [current, *current.parents]
        if (path / "src" / "check_missing_value.py").is_file()
    ),
    None
)

if ROOT_DIR is None:
    raise FileNotFoundError(
        "check_missing_value.py를 찾을 수 없습니다. "
        "프로젝트 폴더 위치를 확인하세요."
    )

sys.path.insert(0, str(ROOT_DIR))

from src.check_missing_value import check_missing

print("프로젝트 루트:", ROOT_DIR)

프로젝트 루트: /Users/2udays/Documents/SUMIN/2026/2026_KNewDeal_AI_Data_Analysis/GummyBearing


In [2]:
folder = ROOT_DIR / "DATA" / "IMS_Bearing_Dataset"

test_folders = ["1st_test", "2nd_test", "3rd_test"]

# 전체 결과 저장
results = []
errors = []

# ==============================
# 1. 전체 파일 결측치 검사
# ==============================

for test_name in test_folders:

    folder_path = folder / test_name

    files = sorted(
        path for path in folder_path.iterdir()
        if path.is_file()
    )

    print(f"[{test_name}] 검사할 파일 수: {len(files)}")

    for i, file_path in enumerate(files, start=1):

        try:
            # 결측치 검사
            result = check_missing(file_path)

            # 결과에 테스트 이름 추가
            result["test_name"] = test_name

            results.append(result)

        except Exception as e:

            errors.append({
                "test_name": test_name,
                "file_path": str(file_path),
                "error": str(e)
            })



# ==============================
# 2. 검사 결과 DataFrame 생성
# ==============================

summary_df = pd.DataFrame(
    results,
    columns=[
        "test_name",
        "file_path",
        "rows",
        "columns",
        "missing_count",
        "has_missing",
        "missing_by_column",
    ],
)

error_df = pd.DataFrame(
    errors,
    columns=[
        "test_name",
        "file_path",
        "error"
    ]
)

missing_files = summary_df[
    summary_df["missing_count"] > 0
]


# ==============================
# 3. 전체 결과 출력
# ==============================

print("\n========== 전체 결측치 검사 결과 ==========")

print("검사 성공 파일 수:", len(summary_df))
print("읽기 실패 파일 수:", len(error_df))
print("결측값이 있는 파일 수:", len(missing_files))
print("전체 결측값 개수:", summary_df["missing_count"].sum())


print("\n[데이터 구조별 파일 수]")
display(
    summary_df.groupby(["test_name", "rows", "columns"])
    .size()
    .reset_index(name="file_count")
)

[1st_test] 검사할 파일 수: 2156
[2nd_test] 검사할 파일 수: 984
[3rd_test] 검사할 파일 수: 6324

========== 전체 결측치 검사 결과 ==========
검사 성공 파일 수: 9464
읽기 실패 파일 수: 0
결측값이 있는 파일 수: 0
전체 결측값 개수: 0

[데이터 구조별 파일 수]


,test_name,rows,columns,file_count
0,1st_test,20480,8,2156
1,2nd_test,20480,4,984
2,3rd_test,20480,4,6324
